# apache_tvm_ffi — smoke test

**Apache TVM FFI** (`apache-tvm-ffi`, imported as `tvm_ffi`) is an ABI-stable C/C++ function and tensor-exchange layer. The examples check the installed version, call global functions across the Python/C++ boundary, and pass a CUDA tensor through DLPack and back.

## Example 1 — the installed version and library (no GPU)

The installed distribution reports the same version as the module, and the shared library that `tvm_ffi` loaded is its core library (`libtvm_ffi.so`, `libtvm_ffi.dylib` or `tvm_ffi.dll`).


In [ ]:
# Adapted from apache/tvm-ffi (Apache-2.0) tests/python/test_libinfo.py.
import importlib.metadata as im, os
from packaging.version import Version
import tvm_ffi

version = Version(tvm_ffi.__version__)
assert str(version) == im.version("apache-tvm-ffi"), (tvm_ffi.__version__, im.version("apache-tvm-ffi"))

lib_path = tvm_ffi.LIB._name
assert os.path.basename(lib_path).split(".")[0] in ("libtvm_ffi", "tvm_ffi"), lib_path
print(f"tvm_ffi {version} from {lib_path}")

## Example 2 — global functions across the boundary (no GPU)

In [ ]:
# Adapted from apache/tvm-ffi (Apache-2.0) tests/python/test_function.py::test_global_func.
import tvm_ffi

@tvm_ffi.register_global_func("nb.scale")
def scale(x, k):
    return x * k

try:
    f = tvm_ffi.get_global_func("nb.scale")
    assert f.same_as(scale) and f(7, 6) == 42
    assert "nb.scale" in tvm_ffi.registry.list_global_func_names()

    add_one = tvm_ffi.get_global_func("testing.add_one")   # defined in C++
    apply = tvm_ffi.get_global_func("testing.apply")       # C++ that calls its first argument
    assert add_one(41) == 42
    assert apply(scale, 2.5, 4) == 10.0                    # C++ -> Python -> C++

    echo = tvm_ffi.get_global_func("testing.echo")
    arr, mp = echo(tvm_ffi.Array([1, "two", 3.0])), echo(tvm_ffi.Map({"a": 1, "b": 2}))
    assert list(arr) == [1, "two", 3.0] and dict(mp.items()) == {"a": 1, "b": 2}
finally:
    tvm_ffi.registry.remove_global_func("nb.scale")
assert tvm_ffi.get_global_func("nb.scale", allow_missing=True) is None

try:  # a Python exception raised inside a callback that C++ invoked
    tvm_ffi.get_global_func("testing.apply")(lambda: {}["from python"])
    raise AssertionError("the Python exception must propagate back through C++")
except KeyError as error:
    assert "from python" in str(error), error
print("python<->C++ global functions and containers round-trip; a Python error survives C++")

## Example 3 — a torch CUDA tensor through DLPack and back

In [ ]:
# Adapted from apache/tvm-ffi (Apache-2.0) tests/python/test_tensor.py::test_tensor_auto_dlpack.
import torch
import tvm_ffi

assert torch.cuda.is_available(), f"requires-gpu cell ran but torch {torch.__version__} sees no CUDA device"

x = torch.arange(1024, dtype=torch.float32, device="cuda")
t = tvm_ffi.from_dlpack(x)
assert tuple(t.shape) == (1024,) and t.device.dlpack_device_type() == tvm_ffi.DLDeviceType.kDLCUDA
back = torch.from_dlpack(t)
assert back.data_ptr() == x.data_ptr()

y = tvm_ffi.get_global_func("testing.echo")(x)
assert isinstance(y, torch.Tensor) and y.data_ptr() == x.data_ptr() and y.device == x.device
back.add_(1)  # shared memory: the write is visible through every view
torch.testing.assert_close(y, torch.arange(1, 1025, dtype=torch.float32, device="cuda"))
print(f"torch {x.device} tensor round-tripped zero-copy through tvm_ffi ({t.device})")